This analysis creates a presentation-ready information card for Valve using the Steam dataset.

The card reports:

- number of Valve games;
- average and median price;
- dominant genre;
- game with the highest positive-rating percentage.

The Publisher Profile is generated from the computed results so that its interpretation remains grounded in the dataset.

In [5]:
import pandas as pd
from IPython.display import HTML, display


STEAM_PATH = "../../data/steam/steam.csv"

steam = pd.read_csv(STEAM_PATH)


# Identify Valve games.
valve = steam[
    steam["publisher"]
    .fillna("")
    .str.split(";")
    .apply(
        lambda x: any(
            p.strip().casefold() == "valve"
            for p in x
        )
    )
].copy()


# Basic statistics.
game_count = valve["appid"].nunique()
mean_price = valve["price"].mean()
median_price = valve["price"].median()


# Dominant genre.
genres = (
    valve[["appid", "genres"]]
    .assign(
        genre=valve["genres"].str.split(";")
    )
    .explode("genre")
)

genres["genre"] = genres["genre"].str.strip()

genre_counts = (
    genres
    .drop_duplicates(["appid", "genre"])
    ["genre"]
    .value_counts()
)

dominant_genre = genre_counts.index[0]
dominant_count = genre_counts.iloc[0]
genre_share = dominant_count / game_count


# Highest-rated Valve game.
valve["total_ratings"] = (
    valve["positive_ratings"]
    + valve["negative_ratings"]
)

rated = valve[
    valve["total_ratings"] > 0
].copy()

rated["positive_pct"] = (
    rated["positive_ratings"]
    / rated["total_ratings"]
    * 100
)

featured = (
    rated
    .sort_values(
        ["positive_pct", "total_ratings"],
        ascending=False,
    )
    .iloc[0]
)


print("Valve games:", game_count)
print("Average price:", f"${mean_price:.2f}")
print("Median price:", f"${median_price:.2f}")
print(
    "Dominant genre:",
    dominant_genre,
    f"({dominant_count} games)",
)
print(
    "Highest-rated game:",
    featured["name"],
    f"({featured['positive_pct']:.2f}%)",
)

Valve games: 30
Average price: $4.48
Median price: $3.99
Dominant genre: Action (26 games)
Highest-rated game: Portal 2 (98.65%)


In [6]:
import pandas as pd
from IPython.display import HTML, display


STEAM_PATH = "../../data/steam/steam.csv"

steam = pd.read_csv(STEAM_PATH)


# Identify Valve games.
valve = steam[
    steam["publisher"]
    .fillna("")
    .str.split(";")
    .apply(
        lambda x: any(
            p.strip().casefold() == "valve"
            for p in x
        )
    )
].copy()


# Basic statistics.
game_count = valve["appid"].nunique()
mean_price = valve["price"].mean()
median_price = valve["price"].median()


# Dominant genre.
genres = (
    valve[["appid", "genres"]]
    .assign(
        genre=valve["genres"].str.split(";")
    )
    .explode("genre")
)

genres["genre"] = genres["genre"].str.strip()

genre_counts = (
    genres
    .drop_duplicates(["appid", "genre"])
    ["genre"]
    .value_counts()
)

dominant_genre = genre_counts.index[0]
dominant_count = genre_counts.iloc[0]
genre_share = dominant_count / game_count


# Highest-rated Valve game.
valve["total_ratings"] = (
    valve["positive_ratings"]
    + valve["negative_ratings"]
)

rated = valve[
    valve["total_ratings"] > 0
].copy()

rated["positive_pct"] = (
    rated["positive_ratings"]
    / rated["total_ratings"]
    * 100
)

featured = (
    rated
    .sort_values(
        ["positive_pct", "total_ratings"],
        ascending=False,
    )
    .iloc[0]
)


print("Valve games:", game_count)
print("Average price:", f"${mean_price:.2f}")
print("Median price:", f"${median_price:.2f}")
print(
    "Dominant genre:",
    dominant_genre,
    f"({dominant_count} games)",
)
print(
    "Highest-rated game:",
    featured["name"],
    f"({featured['positive_pct']:.2f}%)",
)

publisher_profile = (
    f"Valve has {game_count} games in the dataset, "
    f"with an average price of ${mean_price:.2f} "
    f"and a median price of ${median_price:.2f}. "
    f"{dominant_genre} is the dominant genre, representing "
    f"{genre_share:.0%} of Valve games. "
    f"{featured['name']} has the strongest player reception "
    f"at {featured['positive_pct']:.2f}% positive ratings "
    f"across {int(featured['total_ratings']):,} ratings."
)

Valve games: 30
Average price: $4.48
Median price: $3.99
Dominant genre: Action (26 games)
Highest-rated game: Portal 2 (98.65%)


In [7]:
card_html = f"""
<div style="
    background:#f5f5f5;
    border-radius:12px;
    overflow:hidden;
    font-family:Georgia,serif;
">
    <div style="
        background:#2f3b52;
        color:white;
        padding:18px 24px;
    ">
        <h2 style="margin:0;">Valve Publisher Profile</h2>
    </div>

    <table style="
        width:100%;
        border-collapse:collapse;
    ">
        <tr style="background:#eeeeee;">
            <td style="padding:10px 20px;"><b>Games</b></td>
            <td>{game_count}</td>
        </tr>
        <tr>
            <td style="padding:10px 20px;"><b>Average price</b></td>
            <td>${mean_price:.2f}</td>
        </tr>
        <tr style="background:#eeeeee;">
            <td style="padding:10px 20px;"><b>Median price</b></td>
            <td>${median_price:.2f}</td>
        </tr>
        <tr>
            <td style="padding:10px 20px;"><b>Dominant genre</b></td>
            <td>{dominant_genre} ({dominant_count} games)</td>
        </tr>
    </table>

    <div style="
        margin:18px;
        padding:14px;
        background:#e8f5e9;
        border-radius:8px;
    ">
        <b>Featured Game</b><br>
        {featured["name"]}<br>
        {featured["positive_pct"]:.2f}% positive ratings
    </div>

    <div style="padding:0 20px 18px;">
        <h3>Publisher Profile</h3>
        <p>{publisher_profile}</p>
    </div>

    <div style="
        background:#eeeeee;
        color:#777;
        padding:10px 20px;
        font-size:12px;
        font-style:italic;
        text-align:right;
    ">
        Source: Steam Store Games dataset
    </div>
</div>
"""

display(HTML(card_html))

Games,30
Average price,$4.48
Median price,$3.99
Dominant genre,Action (26 games)


The card uses only computed dataset results.

Its pricing interpretation depends on the observed mean and median, its portfolio description depends on the dominant genre's share of Valve games, and its player-reception statement reports the featured game's calculated positive-rating percentage and rating count.